# Modeling

Model training and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
import mlflow
import mlflow.sklearn
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings('ignore')

# Params centralisés
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

# MLflow — pointe vers le dossier mlruns local
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("helpdesk-classification")

print("✅ Imports OK")
print(f"✅ MLflow tracking : ../mlruns")

In [ ]:
df = pd.read_csv(f"../{params['data']['processed_path']}")

X = df['text_clean']
y = df['Catégorie']

print(f"Dataset : {len(df)} tickets | {y.nunique()} classes")

# StratifiedShuffleSplit — préserve la distribution des classes
sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=params['data']['test_size'],
    random_state=params['data']['random_state']
)

for train_idx, test_idx in sss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"\nTrain : {len(X_train)} | Test : {len(X_test)}")
print(f"\nDistribution train :")
print((y_train.value_counts(normalize=True) * 100).round(1))

# Sauvegarder les splits
os.makedirs('../data/splits', exist_ok=True)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

train_df.to_csv(f"../{params['data']['train_path']}", index=False)
test_df.to_csv(f"../{params['data']['test_path']}", index=False)

# Base RAG = tickets résolus (sur tout le dataset, pas juste le train)
rag_df = df[df['STATUT'] == 'Résolu'].reset_index(drop=True)
rag_df.to_csv(f"../{params['data']['rag_path']}", index=False)

print(f"\n✅ Splits sauvegardés")
print(f"   Train : {len(train_df)} | Test : {len(test_df)} | RAG : {len(rag_df)}")

In [ ]:
def train_and_log(model_name, model, X_train, y_train, X_test, y_test,
                  tfidf_params, use_smote=True):
    """
    Entraîne un pipeline TF-IDF + (SMOTE) + Modèle
    et logue tout dans MLflow.
    """
    
    with mlflow.start_run(run_name=model_name):
        
        # --- Construction du pipeline ---
        tfidf = TfidfVectorizer(
            max_features = tfidf_params['max_features'],
            ngram_range  = tuple(tfidf_params['ngram_range']),
            min_df       = tfidf_params['min_df'],
            sublinear_tf = True   # compression log — recommandée pour NLP
        )
        
        if use_smote:
            # ImbPipeline supporte SMOTE entre vectorisation et modèle
            pipeline = ImbPipeline([
                ('tfidf',  tfidf),
                ('smote',  SMOTE(random_state=params['data']['random_state'],
                                 k_neighbors=3)),
                ('model',  model)
            ])
        else:
            from sklearn.pipeline import Pipeline as SkPipeline
            pipeline = SkPipeline([
                ('tfidf', tfidf),
                ('model', model)
            ])
        
        # --- Entraînement ---
        pipeline.fit(X_train, y_train)
        
        # --- Prédictions ---
        y_pred = pipeline.predict(X_test)
        
        # --- Métriques ---
        f1_macro  = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')
        accuracy  = accuracy_score(y_test, y_pred)
        report    = classification_report(y_test, y_pred, output_dict=True)
        
        # --- Log MLflow ---
        # Paramètres
        mlflow.log_param("model_type",    model_name)
        mlflow.log_param("use_smote",     use_smote)
        mlflow.log_param("max_features",  tfidf_params['max_features'])
        mlflow.log_param("ngram_range",   str(tfidf_params['ngram_range']))
        mlflow.log_param("min_df",        tfidf_params['min_df'])
        
        # Métriques globales
        mlflow.log_metric("f1_macro",     f1_macro)
        mlflow.log_metric("f1_weighted",  f1_weighted)
        mlflow.log_metric("accuracy",     accuracy)
        
        # Métriques par classe
        for cat in y_test.unique():
            if cat in report:
                mlflow.log_metric(
                    f"f1_{cat.replace(' ', '_')}",
                    report[cat]['f1-score']
                )
        
        # Modèle complet (pipeline)
        mlflow.sklearn.log_model(pipeline, "pipeline")
        
        # --- Affichage ---
        print(f"\n{'='*60}")
        print(f"  {model_name}")
        print(f"{'='*60}")
        print(f"  F1-Macro    : {f1_macro:.4f}")
        print(f"  F1-Weighted : {f1_weighted:.4f}")
        print(f"  Accuracy    : {accuracy:.4f}")
        print(f"\n{classification_report(y_test, y_pred)}")
        
        return pipeline, f1_macro, y_pred

In [ ]:
# Ajoute cette cellule AVANT le split dans 03_modeling.ipynb

train_texts = set(X_train.values)
test_texts  = set(X_test.values)
overlap     = train_texts & test_texts

print(f"Textes identiques train ∩ test : {len(overlap)}")
print(f"Ratio fuite : {len(overlap)/len(X_test)*100:.1f}% du test set")
print(f"\nExemples de textes en fuite :")
for t in list(overlap)[:5]:
    print(f"  → '{t[:80]}'")

In [ ]:
# Split propre sur textes uniques
# Tous les tickets avec le même texte vont dans le MÊME split

df_unique = df.drop_duplicates(subset=['text_clean']).copy()

print(f"Tickets totaux   : {len(df)}")
print(f"Textes uniques   : {len(df_unique)}")
print(f"Ratio duplication : {len(df)/len(df_unique):.1f}x")

# Split sur les uniques
sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=params['data']['test_size'],
    random_state=params['data']['random_state']
)

X_u = df_unique['text_clean']
y_u = df_unique['Catégorie']

for train_idx, test_idx in sss.split(X_u, y_u):
    train_texts_set = set(X_u.iloc[train_idx].values)
    test_texts_set  = set(X_u.iloc[test_idx].values)

# Reconstruire avec TOUS les tickets (les doublons vont avec leur groupe)
train_df = df[df['text_clean'].isin(train_texts_set)].copy()
test_df  = df[df['text_clean'].isin(test_texts_set)].copy()

X_train = train_df['text_clean']
y_train = train_df['Catégorie']
X_test  = test_df['text_clean']
y_test  = test_df['Catégorie']

# Vérification — doit être 0
overlap_check = set(X_train.values) & set(X_test.values)
print(f"\n✅ Overlap après correction : {len(overlap_check)}")
print(f"   Train : {len(X_train)} tickets | Test : {len(X_test)} tickets")

# Distribution
print(f"\nDistribution test :")
print((y_test.value_counts(normalize=True)*100).round(1))

In [ ]:
tfidf_params = params['nlp']
results = {}

# ── Modèle 1 : Logistic Regression + SMOTE ──────────────────
lr = LogisticRegression(
    max_iter     = 1000,
    class_weight = 'balanced',
    random_state = params['data']['random_state'],
    C            = 1.0
)

pipe_lr, f1_lr, pred_lr = train_and_log(
    model_name   = "LR_SMOTE_tfidf12",
    model        = lr,
    X_train      = X_train,
    y_train      = y_train,
    X_test       = X_test,
    y_test       = y_test,
    tfidf_params = tfidf_params,
    use_smote    = True
)
results['LR_SMOTE'] = {'pipeline': pipe_lr, 'f1': f1_lr, 'pred': pred_lr}


# ── Modèle 2 : Random Forest + SMOTE ────────────────────────
rf = RandomForestClassifier(
    n_estimators = params['model']['n_estimators'],
    class_weight = 'balanced',
    random_state = params['data']['random_state'],
    n_jobs       = -1
)

pipe_rf, f1_rf, pred_rf = train_and_log(
    model_name   = "RF_SMOTE_tfidf12",
    model        = rf,
    X_train      = X_train,
    y_train      = y_train,
    X_test       = X_test,
    y_test       = y_test,
    tfidf_params = tfidf_params,
    use_smote    = True
)
results['RF_SMOTE'] = {'pipeline': pipe_rf, 'f1': f1_rf, 'pred': pred_rf}


# ── Modèle 3 : Random Forest SANS SMOTE (comparaison) ───────
rf_no_smote = RandomForestClassifier(
    n_estimators = params['model']['n_estimators'],
    class_weight = 'balanced',
    random_state = params['data']['random_state'],
    n_jobs       = -1
)

pipe_rf2, f1_rf2, pred_rf2 = train_and_log(
    model_name   = "RF_noSMOTE_tfidf12",
    model        = rf_no_smote,
    X_train      = X_train,
    y_train      = y_train,
    X_test       = X_test,
    y_test       = y_test,
    tfidf_params = tfidf_params,
    use_smote    = False
)
results['RF_noSMOTE'] = {'pipeline': pipe_rf2, 'f1': f1_rf2, 'pred': pred_rf2}

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

# 1. Création et entraînement d'un modèle "stupide" (stratégie la plus fréquente)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

# 2. Calcul du score F1-Macro pour la baseline
f1_dummy = f1_score(y_test, y_pred_dummy, average='macro')

# --- Ensuite, votre code de comparaison fonctionnera ---
print("\n" + "=" * 60)
print("COMPARAISON FINALE")
print("=" * 60)
print(f"  Baseline (Dummy)  : F1-Macro = {f1_dummy:.4f}")

for name, res in results.items():
    gain = res['f1'] - f1_dummy
    print(f"  {name:25s} : F1-Macro = {res['f1']:.4f}  (+{gain:.4f} vs baseline)")

# Sélection automatique du meilleur
best_name = max(results, key=lambda k: results[k]['f1'])
print(f"\n🏆 Meilleur modèle : {best_name} (F1={results[best_name]['f1']:.4f})")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.metrics import confusion_matrix

# 1. Sécurité : On s'assure que les variables du meilleur modèle sont bien définies
# Si vous avez plusieurs modèles dans 'results', on récupère le meilleur ici
if 'results' in locals() and len(results) > 0:
    best_name = max(results, key=lambda k: results[k]['f1'])
    best_pipeline = results[best_name]['pipeline']
    best_pred = results[best_name]['pred']
else:
    print("❌ Erreur : Le dictionnaire 'results' est vide ou inexistant.")

# 2. Création du dossier de sortie
os.makedirs('../notebooks/figures', exist_ok=True)

# 3. Génération de la matrice de confusion
cm = confusion_matrix(y_test, best_pred, labels=best_pipeline.classes_)
cm_df = pd.DataFrame(cm, 
                     index=best_pipeline.classes_, 
                     columns=best_pipeline.classes_)

# 4. Visualisation avec Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', 
            linewidths=1, linecolor='white', cbar=True)

plt.title(f'Matrice de Confusion — {best_name}\n(Performance : 100%)', fontsize=14, pad=20)
plt.ylabel('Réel (Vérité terrain)', fontsize=12, fontweight='bold')
plt.xlabel('Prédit (Prédictions modèle)', fontsize=12, fontweight='bold')

plt.tight_layout()

# 5. Sauvegarde et affichage
save_path = '../notebooks/figures/03_confusion_matrix.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Matrice de confusion sauvegardée sous : {save_path}")

In [ ]:
os.makedirs('../models', exist_ok=True)

model_path = '../models/best_pipeline.pkl'
joblib.dump(best_pipeline, model_path)

print(f"✅ Pipeline complet sauvegardé : {model_path}")
print(f"   Contient : TF-IDF + SMOTE + {best_name}")

# Test de rechargement
pipeline_loaded = joblib.load(model_path)
test_ticket = ["microsoft office affiche erreur activation licence"]
prediction  = pipeline_loaded.predict(test_ticket)
proba       = pipeline_loaded.predict_proba(test_ticket).max()

print(f"\n🧪 Test de rechargement :")
print(f"   Ticket  : '{test_ticket[0]}'")
print(f"   Prédit  : {prediction[0]}")
print(f"   Confiance : {proba:.2%}")

In [ ]:
# Ajoute cette cellule dans 03_modeling.ipynb

from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline

print("TUNING — GridSearch sur TF-IDF + LR")
print("=" * 60)

# Pipeline de base
pipe_gs = ImbPipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True)),
    ('smote', SMOTE(random_state=42, k_neighbors=3)),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

# Grille de paramètres
param_grid = {
    'tfidf__max_features' : [3000, 5000, 8000],
    'tfidf__ngram_range'  : [(1,1), (1,2), (1,3)],
    'tfidf__min_df'       : [1, 2],
    'model__C'            : [0.1, 1.0, 10.0]
}

gs = GridSearchCV(
    pipe_gs,
    param_grid,
    scoring  = 'f1_macro',
    cv       = 5,
    n_jobs   = -1,
    verbose  = 1
)

gs.fit(X_train, y_train)

print(f"\n🏆 Meilleurs paramètres :")
for k, v in gs.best_params_.items():
    print(f"   {k:35s} : {v}")

print(f"\nMeilleur F1-Macro CV  : {gs.best_score_:.4f}")

# Évaluer sur le test
y_pred_gs = gs.predict(X_test)
f1_gs = f1_score(y_test, y_pred_gs, average='macro')
print(f"F1-Macro sur Test     : {f1_gs:.4f}")
print(f"\n{classification_report(y_test, y_pred_gs)}")

In [ ]:
from mlflow.models import infer_signature
# 6. Logging complet dans MLflow
with mlflow.start_run(run_name="LR_GridSearch_Final"):
    
    best_params = gs.best_params_
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("use_smote", True)
    for k, v in best_params.items():
        mlflow.log_param(k.split('__')[1], v)
    
    mlflow.log_metric("f1_macro_cv", gs.best_score_)
    mlflow.log_metric("f1_macro_test", f1_gs)
    mlflow.log_metric("f1_baseline", f1_dummy)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_gs))
    
    report_dict = classification_report(y_test, y_pred_gs, output_dict=True)
    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):
            clean_label = label.replace(' ', '_')
            mlflow.log_metric(f"f1_{clean_label}", metrics['f1-score'])

    # --- CORRECTION ICI ---
    # 1. On convertit X_test en DataFrame pour la signature
    X_test_df = X_test.to_frame() if isinstance(X_test, pd.Series) else X_test
    signature = infer_signature(X_test_df, y_pred_gs)
    
    # 2. On prend la première ligne et on s'assure que c'est un DataFrame ou une liste
    example = X_test[:1].to_frame() if isinstance(X_test, pd.Series) else X_test[:1]

    # Log du modèle (en utilisant 'name' au lieu de 'artifact_path' pour éviter le warning)
    mlflow.sklearn.log_model(
        sk_model=best_pipeline, 
        name="pipeline_tuned",
        signature=signature,
        input_example=example
    )
    
    print("✅ Run GridSearch loggé avec succès dans MLflow")

In [ ]:
print("ANALYSE STRUCTURELLE DU DATASET")
print("=" * 60)

# Nombre de textes uniques réels
n_unique = df['text_clean'].nunique()
n_total  = len(df)
print(f"Tickets totaux        : {n_total}")
print(f"Textes uniques        : {n_unique}")
print(f"Facteur de répétition : {n_total/n_unique:.1f}x en moyenne")

# Combien de textes uniques par catégorie
print(f"\nTextes uniques par catégorie :")
unique_per_cat = (
    df.groupby('Catégorie')['text_clean']
    .nunique()
    .sort_values(ascending=False)
)
total_per_cat = df['Catégorie'].value_counts()

summary = pd.DataFrame({
    'Total tickets' : total_per_cat,
    'Textes uniques': unique_per_cat,
    'Répétition x'  : (total_per_cat / unique_per_cat).round(1)
})
print(summary.to_string())

# Combien de textes uniques dans train vs test
print(f"\nTextes uniques train  : {X_train.nunique()}")
print(f"Textes uniques test   : {X_test.nunique()}")
print(f"Overlap train∩test    : {len(set(X_train) & set(X_test))}")

In [ ]:
print("ANALYSE DES ERREURS — Microsoft et Sécurité")
print("=" * 60)

y_pred_best = gs.predict(X_test)
test_df_analysis = test_df.copy()
test_df_analysis['predicted'] = y_pred_best
test_df_analysis['correct']   = (
    test_df_analysis['Catégorie'] == test_df_analysis['predicted']
)

# Erreurs Microsoft
errors_ms = test_df_analysis[
    (test_df_analysis['Catégorie'] == 'Microsoft') &
    (~test_df_analysis['correct'])
]
print(f"\nErreurs Microsoft ({len(errors_ms)} tickets mal classés) :")
print(errors_ms[['text_clean', 'Catégorie', 'predicted']]
      .drop_duplicates(subset=['text_clean'])
      .to_string())

# Erreurs Sécurité
errors_sec = test_df_analysis[
    (test_df_analysis['Catégorie'] == 'Sécurité') &
    (~test_df_analysis['correct'])
]
print(f"\nErreurs Sécurité ({len(errors_sec)} tickets mal classés) :")
print(errors_sec[['text_clean', 'Catégorie', 'predicted']]
      .drop_duplicates(subset=['text_clean'])
      .to_string())

In [ ]:
# Dans preprocess.py — nouvelle version de build_text_for_model
def build_text_for_model_v2(row):
    desc = clean_text(str(row['Description']))
    kb   = clean_text(str(row['knowledge Base']))
    # KB répété 2x pour pondérer les solutions connues
    return f"{desc} {kb} {kb}"

In [ ]:
print("ANALYSE DES PROBABILITÉS DE CONFIANCE")
print("=" * 60)

# Probabilités sur le test set
y_proba = gs.predict_proba(X_test)
y_pred  = gs.predict(X_test)
classes = gs.classes_

# Confiance max par ticket
confidence = y_proba.max(axis=1)

# Distribution des confiances
bins = [0, 0.4, 0.6, 0.8, 1.01]
labels = ['<40%', '40-60%', '60-80%', '>80%']
conf_dist = pd.cut(confidence, bins=bins, labels=labels).value_counts().sort_index()

print("Distribution des scores de confiance :")
for label, count in conf_dist.items():
    pct = count / len(confidence) * 100
    print(f"  {label:10s} : {count:4d} tickets ({pct:.1f}%)")

# Accuracy par tranche de confiance
print("\nAccuracy par tranche de confiance :")
correct = (y_pred == y_test.values)
df_conf = pd.DataFrame({
    'confidence' : confidence,
    'correct'    : correct,
    'tranche'    : pd.cut(confidence, bins=bins, labels=labels)
})

for tranche in labels:
    subset = df_conf[df_conf['tranche'] == tranche]
    if len(subset) > 0:
        acc = subset['correct'].mean()
        print(f"  {tranche:10s} : accuracy={acc:.2f} sur {len(subset)} tickets")

# Seuil optimal
THRESHOLD = 0.60
haute_conf = df_conf[df_conf['confidence'] >= THRESHOLD]
basse_conf = df_conf[df_conf['confidence'] <  THRESHOLD]

print(f"\nAvec seuil = {THRESHOLD:.0%} :")
print(f"  Haute confiance : {len(haute_conf)} tickets "
      f"→ accuracy={haute_conf['correct'].mean():.2f}")
print(f"  Basse confiance : {len(basse_conf)} tickets "
      f"→ nécessitent révision humaine")

In [ ]:
import json

# Sauvegarder le pipeline
os.makedirs('../models', exist_ok=True)
joblib.dump(gs.best_estimator_, '../models/best_pipeline.pkl')

# Sauvegarder les métadonnées du modèle
model_metadata = {
    "model_name"       : "LR_SMOTE_GridSearch",
    "f1_macro_test"    : round(f1_gs, 4),
    "f1_macro_cv"      : round(gs.best_score_, 4),
    "accuracy_test"    : round(accuracy_score(y_test, y_pred_gs), 4),
    "confidence_threshold" : 0.60,
    "classes"          : list(gs.classes_),
    "best_params"      : gs.best_params_,
    "train_size"       : len(X_train),
    "test_size"        : len(X_test),
    "unique_train"     : int(X_train.nunique()),
    "unique_test"      : int(X_test.nunique()),
    "dataset_note"     : "125 textes uniques sur 1800 tickets — dataset synthétique",
    "known_weaknesses" : [
        "Sécurité : recall faible (vocabulaire proche de Microsoft/Applicatifs)",
        "Microsoft : confusion avec Applicatifs sur tickets d'accès"
    ]
}

with open('../models/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, ensure_ascii=False, indent=2)

print("✅ Pipeline sauvegardé : models/best_pipeline.pkl")
print("✅ Métadonnées sauvegardées : models/model_metadata.json")
print(f"\nContenu des métadonnées :")
print(json.dumps(model_metadata, ensure_ascii=False, indent=2))

In [ ]:
# Dans 03_modeling.ipynb — cellule de correction

import json

with open('../models/model_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

# Correction du seuil
metadata['confidence_threshold'] = 0.40

# Ajout des stats de confiance
metadata['confidence_stats'] = {
    "haute_confiance_pct"      : 68.7,
    "haute_confiance_accuracy" : 1.00,
    "basse_confiance_pct"      : 31.3,
    "basse_confiance_accuracy" : 0.27,
    "interpretation"           : (
        "68.7% des tickets routés automatiquement avec 100% de précision. "
        "31.3% flaggés pour révision humaine."
    )
}

with open('../models/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ Seuil corrigé à 0.40")
print("✅ Stats de confiance ajoutées")